In [1]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Sequence, Tuple

import pandas as pd


@dataclass(frozen=True)
class MesPaths:
    disease_csv: Path
    annot_csv: Path
    secretion_db_json: Path
    genome_ko_dir: Path
    uptake_pred_csv: Path
    out_dir: Path


def _first_existing_col(df: pd.DataFrame, candidates: Sequence[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"None of columns exist: {candidates}. Available: {list(df.columns)[:30]}..."
    )


def load_and_balance_disease_data(
    disease_csv: Path,
    *,
    disease_col_candidates: Sequence[str] = ("disease",),
    sample_id_col_candidates: Sequence[str] = ("X", "Sample", "sample", "sample_id"),
    min_case_n: int = 50,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, str, str]:
    """Returns (disease_df, healthy_df, disease_col, sample_id_col)."""
    df = pd.read_csv(disease_csv)
    disease_col = _first_existing_col(df, disease_col_candidates)
    sample_id_col = _first_existing_col(df, sample_id_col_candidates)

    disease_df = df[df[disease_col] != 0].copy()
    healthy_df = df[df[disease_col] == 0].copy()

    kept_diseases = [
        k for k, v in disease_df[disease_col].value_counts().items() if v > min_case_n
    ]
    disease_df = disease_df[disease_df[disease_col].isin(kept_diseases)].reset_index(
        drop=True
    )
    healthy_df = healthy_df.reset_index(drop=True)

    if len(disease_df) == 0 or len(healthy_df) == 0:
        raise ValueError(
            f"After filtering, empty disease/healthy set. disease={len(disease_df)} healthy={len(healthy_df)}"
        )

    healthy_df = healthy_df.sample(n=len(disease_df), random_state=random_state).reset_index(
        drop=True
    )
    return disease_df, healthy_df, disease_col, sample_id_col


def find_sample_species(
    df: pd.DataFrame,
    *,
    sample_id_col: str,
    species_cols: Sequence[str],
    abundance_threshold: float = 100.0,
) -> List[Dict[str, List[str]]]:
    """For each sample, return {sample_id: [species with abundance > threshold]}."""
    df_species = df.loc[:, list(species_cols)]
    result: List[Dict[str, List[str]]] = []
    for idx, row in df_species.iterrows():
        present = [c for c in df_species.columns if row[c] > abundance_threshold]
        result.append({str(df.loc[idx, sample_id_col]): present})
    return result


def collect_all_species(sample_species: Iterable[Mapping[str, Sequence[str]]]) -> List[str]:
    s: set[str] = set()
    for d in sample_species:
        species = next(iter(d.values()))
        s.update(species)
    return sorted(s)


def select_representative_genomes(
    annot_df: pd.DataFrame,
    *,
    species_list: Sequence[str],
    top_n: int = 1,
    species_name_col_candidates: Sequence[str] = ("Species", "species", "organism_name"),
    assembly_level_col_candidates: Sequence[str] = ("assembly_level", "assemblyLevel"),
    assembly_accession_col_candidates: Sequence[str] = (
        "#assembly_accession",
        "assembly_accession",
        "GCA",
    ),
    species_taxid_col_candidates: Sequence[str] = ("species_taxid", "taxid"),
) -> pd.DataFrame:
    """Filter annotation table to species_list and pick best assemblies per species_taxid."""
    species_col = _first_existing_col(annot_df, species_name_col_candidates)
    assembly_level_col = _first_existing_col(annot_df, assembly_level_col_candidates)
    accession_col = _first_existing_col(annot_df, assembly_accession_col_candidates)
    taxid_col = _first_existing_col(annot_df, species_taxid_col_candidates)

    df = annot_df[annot_df[species_col].isin(species_list)].copy()
    if len(df) == 0:
        raise ValueError("No genomes found for given species list in annotation table.")

    df[assembly_level_col] = df[assembly_level_col].replace(
        {
            "Contig": 3,
            "Scaffold": 2,
            "Chromosome": 1,
            "Complete Genome": 0,
        }
    )
    df = df.sort_values(by=[species_col, assembly_level_col])
    df = df.groupby(taxid_col, as_index=False).head(top_n)

    df = df.rename(
        columns={
            species_col: "Species",
            assembly_level_col: "assembly_level",
            accession_col: "#assembly_accession",
            taxid_col: "species_taxid",
        }
    )
    return df.reset_index(drop=True)


def load_secretion_db(secretion_db_json: Path) -> Dict[str, List[Dict[str, List[str]]]]:
    """compound -> list of {reaction: [KO,...]} (KO list non-empty)."""
    with open(secretion_db_json, "r") as f:
        compounds2ko = json.load(f)

    c2secretion: Dict[str, List[Dict[str, List[str]]]] = {}
    for c, rn_link in compounds2ko.items():
        kept: List[Dict[str, List[str]]] = []
        for rn2ko in rn_link:
            for rn, ko_links in rn2ko.items():
                if ko_links:
                    kept.append({rn: ko_links})
        if kept:
            c2secretion[c] = kept
    return c2secretion


def predict_secretion_for_content(
    content: str,
    c2secretion: Mapping[str, Sequence[Mapping[str, Sequence[str]]]],
) -> Dict[str, List[str]]:
    pred_result: Dict[str, List[str]] = {}
    for compound, rn_ls in c2secretion.items():
        hits: List[str] = []
        for rn_dict in rn_ls:
            for rn, ko_ls in rn_dict.items():
                if all((ko in content) for ko in ko_ls):
                    hits.append(rn)
        pred_result[compound] = hits
    return pred_result


def build_secretion_matrix(
    genome_ko_dir: Path,
    c2secretion: Mapping[str, Sequence[Mapping[str, Sequence[str]]]],
) -> pd.DataFrame:
    """Scan genome KO annotation files under genome_ko_dir and return binary matrix."""
    f_result: Dict[str, Dict[str, List[str]]] = {}
    for root, _dirs, files in os.walk(genome_ko_dir):
        for fname in files:
            fpath = Path(root) / fname
            with open(fpath, "r") as f:
                content = f.read()
            f_result[fname] = predict_secretion_for_content(content, c2secretion)

    if not f_result:
        raise ValueError(f"No genome KO files found under {genome_ko_dir}")

    compounds = list(next(iter(f_result.values())).keys())
    rows: List[List[int]] = []
    ids: List[str] = []
    for gid, pred in f_result.items():
        ids.append(gid)
        rows.append([1 if pred[c] else 0 for c in compounds])

    df = pd.DataFrame(rows, columns=compounds)
    df["ID"] = ids
    return df


def _to_binary(x: float) -> int:
    return 1 if x > 0.5 else 0


def _mes_for_compounds(
    secretion_binary: pd.DataFrame,
    uptake_binary: pd.DataFrame,
    compound_cols: Sequence[str],
) -> List[float]:
    r: List[float] = []
    for col in compound_cols:
        p = float(secretion_binary[col].sum())
        c = float(uptake_binary[col].sum())
        r.append(0.0 if (p + c) == 0 else (2.0 * p * c / (p + c)))
    return r


def calculate_mes_per_sample(
    sample_species: Iterable[Mapping[str, Sequence[str]]],
    *,
    secretion_df: pd.DataFrame,
    uptake_df: pd.DataFrame,
    genome_map_df: pd.DataFrame,
    compound_cols: Sequence[str],
) -> pd.DataFrame:
    if "ID" not in secretion_df.columns:
        raise KeyError("secretion_df must contain column 'ID'")
    if "Query" not in uptake_df.columns:
        raise KeyError("uptake_df must contain column 'Query'")
    if "Species" not in genome_map_df.columns or "#assembly_accession" not in genome_map_df.columns:
        raise KeyError("genome_map_df must contain 'Species' and '#assembly_accession'")

    result: Dict[str, List[float]] = {}

    species2acc = (
        genome_map_df[["Species", "#assembly_accession"]]
        .dropna()
        .drop_duplicates(subset=["Species"])
        .set_index("Species")["#assembly_accession"]
        .astype(str)
        .to_dict()
    )

    for d in sample_species:
        sample_id = str(next(iter(d.keys())))
        species_ls = list(next(iter(d.values())))
        gca_ls = [species2acc[s] for s in species_ls if s in species2acc]
        if not gca_ls:
            continue

        s_df = secretion_df[secretion_df["ID"].isin(gca_ls)]
        r_df = uptake_df[uptake_df["Query"].isin(gca_ls)]
        if len(s_df) == 0 or len(r_df) == 0:
            continue

        result[sample_id] = _mes_for_compounds(s_df, r_df, compound_cols)

    out = pd.DataFrame.from_dict(result, orient="index", columns=list(compound_cols)).reset_index()
    return out.rename(columns={"index": "Sample"})


def load_uptake_predictions(
    uptake_pred_csv: Path,
    *,
    compound_cols: Sequence[str],
) -> pd.DataFrame:
    df = pd.read_csv(uptake_pred_csv)
    if "Query" not in df.columns:
        raise KeyError("uptake prediction csv must contain column 'Query'")

    kept = [c for c in compound_cols if c in df.columns]
    df = df[kept + ["Query"]].copy()
    for c in kept:
        df[c] = df[c].map(_to_binary)
    return df

In [ ]:
P = MesPaths(
    disease_csv=Path("../data/Oral_metagenome_data.csv"),
    annot_csv=Path("../data/annot_data.csv"),
    secretion_db_json=Path("../data/Synthetic_reaction_db.json"),
    genome_ko_dir=Path("mes_genome_data"), #
    uptake_pred_csv=Path("pred_result.csv"), # predicated by gelato
    out_dir=Path("mes"),
)

NAME2ID = {
    "control": 0,
    "caries": 1,
    "cap": 2,
    "CRC": 3,
    "CD_PD": 4,
    "DM": 5,
    "hypertension": 6,
    "H2RA": 7,
    "Induced gingivitis": 8,
    "MRONJ": 9,
    "mucositis": 10,
    "PD": 11,
    "peri-i mplantitis": 12,
    "pregnancy": 13,
    "PDAC": 14,
    "PPI": 15,
    "RA": 16,
    "Spontaneous gingivitis": 17,
    "ASD": 18,
}
ID2NAME = {v: k for k, v in NAME2ID.items()}

MIN_CASE_N = 50
ABUNDANCE_THRESHOLD = 100.0

disease_df, healthy_df, disease_col, sample_id_col = load_and_balance_disease_data(
    P.disease_csv,
    min_case_n=MIN_CASE_N,
)
species_cols = list(disease_df.columns[5:])

In [3]:
disease_sample_species = []
kept_diseases = []
for d in sorted(disease_df[disease_col].unique()):
    df_d = disease_df[disease_df[disease_col] == d]
    cur = find_sample_species(
        df_d,
        sample_id_col=sample_id_col,
        species_cols=species_cols,
        abundance_threshold=ABUNDANCE_THRESHOLD,
    )
    if len(cur) > 90:
        kept_diseases.append(d)
        disease_sample_species.extend(cur)


if kept_diseases:
    disease_df = disease_df[disease_df[disease_col].isin(kept_diseases)].reset_index(drop=True)

for d in sorted(disease_df[disease_col].unique()):
    df_d = disease_df[disease_df[disease_col] == d]
    cur = find_sample_species(
        df_d,
        sample_id_col=sample_id_col,
        species_cols=species_cols,
        abundance_threshold=ABUNDANCE_THRESHOLD,
    )
    print(f"{ID2NAME.get(int(d), str(d))} sample counts: {len(cur)}")

healthy_sample_species = find_sample_species(
    healthy_df,
    sample_id_col=sample_id_col,
    species_cols=species_cols,
    abundance_threshold=ABUNDANCE_THRESHOLD,
)
print("healthy sample counts:", len(healthy_sample_species))

all_sample_species = disease_sample_species + healthy_sample_species
print("total sample counts:", len(all_sample_species))

all_species = collect_all_species(all_sample_species)
print("total unique species:", len(all_species))

caries sample counts: 246
cap sample counts: 331
hypertension sample counts: 104
H2RA sample counts: 110
PD sample counts: 244
pregnancy sample counts: 98
RA sample counts: 165
healthy sample counts: 1366
total sample counts: 2664
total unique species: 417


In [4]:
annot_df = pd.read_csv(P.annot_csv)

genome_map = select_representative_genomes(
    annot_df,
    species_list=all_species,
    top_n=1,
)

annot_species = set(genome_map["Species"].tolist())
not_annot_species = sorted(set(all_species) - annot_species)

In [5]:
secretion_cache = P.out_dir / "secretion_result.csv"

c2secretion = load_secretion_db(P.secretion_db_json)
compound_cols = list(c2secretion.keys())
if secretion_cache.exists():
    secretion_df = pd.read_csv(secretion_cache)
else:
    secretion_df = build_secretion_matrix(P.genome_ko_dir, c2secretion)
    secretion_df.to_csv(secretion_cache, index=False)
    
    
uptake_df = load_uptake_predictions(P.uptake_pred_csv, compound_cols=compound_cols)
common_compounds = [c for c in compound_cols if (c in secretion_df.columns and c in uptake_df.columns)]
secretion_use = secretion_df[common_compounds + ["ID"]].copy()
uptake_use = uptake_df[common_compounds + ["Query"]].copy()

for d in sorted(disease_df[disease_col].unique()):
    df_d = disease_df[disease_df[disease_col] == d]
    cur_d_species = find_sample_species(
        df_d,
        sample_id_col=sample_id_col,
        species_cols=species_cols,
        abundance_threshold=ABUNDANCE_THRESHOLD,
    )
    mes_df = calculate_mes_per_sample(
        cur_d_species,
        secretion_df=secretion_use,
        uptake_df=uptake_use,
        genome_map_df=genome_map,
        compound_cols=common_compounds,
    )
    out_path = P.out_dir / f"{ID2NAME.get(int(d), str(d))}_mes.csv"
    mes_df.to_csv(out_path, index=False)

healthy_mes = calculate_mes_per_sample(
    healthy_sample_species,
    secretion_df=secretion_use,
    uptake_df=uptake_use,
    genome_map_df=genome_map,
    compound_cols=common_compounds,
)
healthy_out = P.out_dir / "healthy_mes.csv"
healthy_mes.to_csv(healthy_out, index=False)